In [1]:
import os
from tqdm import tqdm
from pyoxigraph import Store, RdfFormat
import torch
import pandas as pd
from openai import OpenAI
import pickle
import json
import numpy as np
from IPython.display import display, Markdown
from copy import deepcopy
import time

from kneed import KneeLocator

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from elasticsearch import Elasticsearch

from utils import ollama_request, EMBEDD_MODEL_1
import warnings

import spacy
nlp = spacy.load("en_core_web_sm")
warnings.filterwarnings("ignore")

from IPython.display import display, HTML

INDEX_NAME_ENHANCED = f"enhanced_{os.getenv('INDEX_NAME')}"
ENT_INDEX_NAME_ENHANCED = f"{INDEX_NAME_ENHANCED}_entities_index"
TRIPLETS_INDEX_NAME_ENHANCED = f"{INDEX_NAME_ENHANCED}_triplets_index"

INDEX_NAME_NOT_ENHANCED = f"not_enhanced_{os.getenv('INDEX_NAME')}"
ENT_INDEX_NAME_NOT_ENHANCED = f"{INDEX_NAME_NOT_ENHANCED}_entities_index"
TRIPLETS_INDEX_NAME_NOT_ENHANCED = f"{INDEX_NAME_NOT_ENHANCED}_triplets_index"

FRAGMENT_INDEX_NAME = f"{os.getenv('INDEX_NAME')}_fragments"

EMBEDDING_MODEL = EMBEDD_MODEL_1

es_client = Elasticsearch('http://localhost:9200')
openai_client = OpenAI()

graph_enhanced = Store()
graph_enhanced.load(
    path=f"{INDEX_NAME_ENHANCED}_triplets.ttl",
    format=RdfFormat.TURTLE
)

graph_not_enhanced = Store()
# graph_not_enhanced.load(
#     path=f"{INDEX_NAME_NOT_ENHANCED}_triplets.ttl",
#     format=RdfFormat.TURTLE
# )
graph_not_enhanced.load(
    path=f"{INDEX_NAME_NOT_ENHANCED}_triplets_1000.ttl",
    format=RdfFormat.TURTLE
)

/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


I removed from prompt to refer to sources by indexes using:

Each fragment contains an index.
You can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".
Use just syntax with [index] to refer to the fact, do not use any additional words.

as it is problematic for worst models to answer the qustion then

In [2]:
# REFERENCE_NUM = 21

# final_triplets = enhanced_final_triplets_dict["main_plus_additional_triplets"]

# display(final_triplets[final_triplets["index"] == REFERENCE_NUM][["subject", "predicate", "object"]].reset_index(drop=True))

# display(Markdown("### Reference Fragment"))
# display(Markdown(get_fragment(final_triplets, REFERENCE_NUM, FRAGMENT_INDEX_NAME)))

# display(Markdown("### Reference Speech"))
# display(Markdown(get_speech(final_triplets, REFERENCE_NUM, os.getenv("INDEX_NAME"))))

In [3]:
def extract_objects(text):
    doc = nlp(text)

    subjects = []
    target_deps = {
        "nsubj",
        "nsubjpass",
        "dobj",
        "pobj",
        "iobj",
    }
    for token in doc:
        if token.dep_ in target_deps:
            subject = " ".join(
                t.text for t in token.subtree
            )
            subjects.append(subject)

    return subjects

In [4]:
def execute_query(graph, start_date, end_date, filtering_criteria):
    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?fragment_id ?speech_id ?triplet_id
WHERE {{
    ?stmt rdf:reifies <<( ?s ?p ?o )>> ;
          ns1:date ?date ;
          ns1:fragment_id ?fragment_id ;
          ns1:triplet_id ?triplet_id ;
          ns1:speech_id ?speech_id .

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
        {filtering_criteria}
    )
}}
""".format(
        start_date=start_date,
        end_date=end_date,
        filtering_criteria=filtering_criteria
    )
    return graph.query(query)

In [5]:
def get_df_from_query_result(query_res):
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "fragment_id": [],
        "speech_id": [],
    }
    for row in query_res:
        fragment_id = row["fragment_id"].value
        speech_id = row["speech_id"].value

        triplets_resp["subject"].append(row["s"].value.split('/')[-1])
        triplets_resp["predicate"].append(row["p"].value.split('/')[-1])
        triplets_resp["object"].append(row["o"].value.split('/')[-1])
        triplets_resp["date"].append(str(row["date"].value))
        triplets_resp["fragment_id"].append(fragment_id)
        triplets_resp["speech_id"].append(speech_id)
    return pd.DataFrame(triplets_resp)

In [6]:
def get_triplets_by_id(
    graph,
    triplet_ids,
    start_date="1900-01-01",
    end_date="2100-01-01"
):
    triplet_ids_values = " ".join(str(x) for x in triplet_ids)

    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?fragment_id ?speech_id ?triplet_id
WHERE {{
    ?stmt rdf:reifies <<( ?s ?p ?o )>> ;
          ns1:date ?date ;
          ns1:fragment_id ?fragment_id ;
          ns1:speech_id ?speech_id ;
          ns1:triplet_id ?triplet_id .

    VALUES ?triplet_id {{ {triplet_ids_values} }}

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
    )
}}
""".format(
        start_date=start_date,
        end_date=end_date,
        triplet_ids_values=triplet_ids_values,
    )
    return graph.query(query)

In [7]:
def knee_cutoff(scores):
    scores = np.sort(np.asarray(scores))[::-1]

    indices = np.arange(len(scores))

    knee = KneeLocator(
        indices,
        scores,
        curve="convex",
        direction="decreasing"
    )
    if not knee.knee:
        return len(scores) // 2
    return int(knee.knee) - 1

In [8]:
def score_df_cutoff(df, top_k=100):
    df = df.sort_values(by="score", ascending=False).iloc[:top_k, :]

    sorted_scores = df["score"].values
    if len(sorted_scores) < 2:
        return df
    gaps = sorted_scores[:-1] - sorted_scores[1:]
    cutoff_1 = np.argmax(gaps) + 1
    cutoff_2 = knee_cutoff(sorted_scores)

    # if cutoff_2 == 0:
    #     cutoff = cutoff_1
    # else:
    #     cutoff = cutoff_2
    cutoff = max(cutoff_1, cutoff_2)
    return df.iloc[:cutoff, :]

In [9]:
def find_similar_entities(entity, ent_index_name, score_threshold=0.5, top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(entity)
    final_res = {
        "score": [],
        "value_name": [],
    }
    response = es_client.search(
        index=ent_index_name,
        size=100,
        knn={
            "field": "value_embedding",
            "query_vector": query_embedding.tolist(),
            "k": 100,
            "num_candidates": 100,
            "filter": {
                "match_all": {}
            }
        },
        source=["value_name"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["value_name"].append(result["_source"]["value_name"])
    
    final_res = pd.DataFrame(final_res)
    return score_df_cutoff(final_res, top_k=top_k)


In [10]:
def find_question_related_triplets(graph, question, triplets_index_name, score_threshold=0.5, start_date="1900-01-01", end_date="2100-01-01", top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(question)
    final_res = {
        "score": [],
        "triplet_id": [],
    }
    response = es_client.search(
        index=triplets_index_name,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": query_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "range": {
                    "date": {
                        "gte": start_date,
                        "lte": end_date
                    }
                }
            }
        },
        source=["triplet_id"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    related_triplets = pd.DataFrame(final_res)
    related_triplets = score_df_cutoff(related_triplets, top_k=top_k)
    related_ids = list(related_triplets["triplet_id"])

    query_res = get_triplets_by_id(graph, related_ids, start_date=start_date, end_date=end_date)
    res_df = get_df_from_query_result(query_res)
    res_df["information_type"] = "main"

    return res_df


In [11]:
def get_triplets_based_on_unique_entities_subgraph(graph, main_triplets_related_to_question, question_embedding, triplets_index_name, score_threshold=0.5, top_triplet_k=10, start_date="1900-01-01", end_date="2100-01-01"):
    filtering_criteria = "&& (?s = entity:{val} || ?o = entity:{val})"
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "fragment_id": [],
        "speech_id": [],
        "triplet_id": [],
    }
    unique_entities = set(main_triplets_related_to_question["subject"].values).union(set(main_triplets_related_to_question["object"].values))
    for ent in unique_entities:
        try:
            query_res = list(
                execute_query(
                    graph=graph,
                    start_date=start_date,
                    end_date=end_date,
                    filtering_criteria=filtering_criteria.format(val=ent),
                )
            )
            # print(f"Found {len(query_res)} triplets for entity: {ent}")
        except Exception as e:
            continue

        for row in query_res:
            fragment_id = row["fragment_id"].value
            speech_id = row["speech_id"].value

            triplet_id = row["triplet_id"].value
            
            triplets_resp["subject"].append(row["s"].value.split('/')[-1])
            triplets_resp["predicate"].append(row["p"].value.split('/')[-1])
            triplets_resp["object"].append(row["o"].value.split('/')[-1])
            triplets_resp["date"].append(str(row["date"].value))
            triplets_resp["fragment_id"].append(fragment_id)
            triplets_resp["speech_id"].append(speech_id)
            triplets_resp["triplet_id"].append(int(triplet_id))

    triplets_df = pd.DataFrame(triplets_resp)
    final_res = {
        "score": [],
        "triplet_id": [],
    }

    response = es_client.search(
        index=triplets_index_name,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": question_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "terms": {
                    "triplet_id": triplets_df["triplet_id"].tolist()
                }
            }
        },
        source=["triplet_id"]
    )
    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    final_scores = pd.DataFrame(final_res)
    final_scores["triplet_id"] = final_scores["triplet_id"].astype("int64")
    triplets_df["triplet_id"] = triplets_df["triplet_id"].astype("int64")


    triplets_df = triplets_df.merge(
        final_scores,
        on="triplet_id",
        how="inner"
    )
    triplets_df.drop_duplicates(subset=["subject", "predicate", "object", "date", "fragment_id", "speech_id"], inplace=True)
    triplets_df = triplets_df.sort_values(by="score", ascending=False)
    triplets_df = triplets_df.iloc[(len(main_triplets_related_to_question)-1):, :]
    triplets_df = score_df_cutoff(triplets_df, top_k=top_triplet_k)
    triplets_df["information_type"] = "additional"
    triplets_df = triplets_df.drop(columns=["score", "triplet_id"])
    return triplets_df

In [12]:
def get_triplets_based_on_unique_entities_main_plus_addtional(graph, unique_entities, question_embedding, triplets_index_name, score_threshold=0.5, top_triplet_k=10, start_date="1900-01-01", end_date="2100-01-01"):
    filtering_criteria = "&& (?s = entity:{val} || ?o = entity:{val})"
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "fragment_id": [],
        "speech_id": [],
        "triplet_id": [],
    } 
    for ent in unique_entities:
        try:
            query_res = list(
                execute_query(
                    graph=graph,
                    start_date=start_date,
                    end_date=end_date,
                    filtering_criteria=filtering_criteria.format(val=ent),
                )
            )
            # print(f"Found {len(query_res)} triplets for entity: {ent}")
        except Exception as e:
            continue

        for row in query_res:
            fragment_id = row["fragment_id"].value
            speech_id = row["speech_id"].value

            triplet_id = row["triplet_id"].value
            
            triplets_resp["subject"].append(row["s"].value.split('/')[-1])
            triplets_resp["predicate"].append(row["p"].value.split('/')[-1])
            triplets_resp["object"].append(row["o"].value.split('/')[-1])
            triplets_resp["date"].append(str(row["date"].value))
            triplets_resp["fragment_id"].append(fragment_id)
            triplets_resp["speech_id"].append(speech_id)
            triplets_resp["triplet_id"].append(int(triplet_id))

    triplets_df = pd.DataFrame(triplets_resp)
    final_res = {
        "score": [],
        "triplet_id": [],
    }

    response = es_client.search(
        index=triplets_index_name,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": question_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "terms": {
                    "triplet_id": triplets_df["triplet_id"].tolist()
                }
            }
        },
        source=["triplet_id"]
    )
    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    final_scores = pd.DataFrame(final_res)
    final_scores["triplet_id"] = final_scores["triplet_id"].astype("int64")
    triplets_df["triplet_id"] = triplets_df["triplet_id"].astype("int64")

    triplets_df = triplets_df.merge(
        final_scores,
        on="triplet_id",
        how="inner"
    )
    triplets_df.drop_duplicates(subset=["subject", "predicate", "object", "date", "fragment_id", "speech_id"], inplace=True)
    triplets_df = score_df_cutoff(triplets_df, top_k=top_triplet_k)
    triplets_df["information_type"] = "additional"
    triplets_df = triplets_df.drop(columns=["score", "triplet_id"])
    return triplets_df

In [13]:
def find_additional_related_triplets(graph, question, triplets_index_name, ent_index_name, score_threshold=0.5, top_entity_k=10, top_triplet_k=10, start_date="1900-01-01", end_date="2100-01-01"):
    question_objects = extract_objects(question)
    question_embedding = EMBEDDING_MODEL.encode(question)
    similar_entities = []
    for object in question_objects:
        similar_entities = similar_entities + list(find_similar_entities(object, ent_index_name=ent_index_name, score_threshold=score_threshold, top_k=top_entity_k)["value_name"].values)

    triplets_df = get_triplets_based_on_unique_entities_main_plus_addtional(
        graph=graph,
        unique_entities=similar_entities,
        question_embedding=question_embedding,
        triplets_index_name=triplets_index_name,
        score_threshold=score_threshold,
        top_triplet_k=top_triplet_k,
        start_date=start_date,
        end_date=end_date
    )

    return triplets_df


In [14]:
def find_subgraph_for_question(graph, main_triplets_related_to_question, question, triplets_index_name, score_threshold=0.5, top_triplet_k=10, start_date="1900-01-01", end_date="2100-01-01"):

    subgraph_df = get_triplets_based_on_unique_entities_subgraph(
        graph=graph,
        main_triplets_related_to_question=main_triplets_related_to_question,
        question_embedding=EMBEDDING_MODEL.encode(question),
        triplets_index_name=triplets_index_name,
        score_threshold=score_threshold,
        top_triplet_k=top_triplet_k,
        start_date=start_date,
        end_date=end_date
    )
    final_df = pd.concat([main_triplets_related_to_question, subgraph_df], ignore_index=True).drop_duplicates(subset=["subject", "predicate", "object", "date", "fragment_id", "speech_id"]).reset_index(drop=True)

    return final_df

In [15]:
def get_main_plus_additional_triplets_prompt(question, main_triplets_df, additional_triplets_df, sort_time=True):
    prompt_template = """
You are a political scientist model. You are given a research question and a set of facts from a knowledge graph.
Each of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.
Each of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.
The triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.

Facts are split into two categories: main and additional.
Main facts (most important) are directly related to the research question. 
Additional facts provide context or background information for entities present in the question.

Main Facts:
{main_facts}

Additional Facts:
{additional_facts}

Research question: {question}

Your task is to answer the research question based on the provided facts.
    """
    final_triplets = pd.concat([main_triplets_df, additional_triplets_df], ignore_index=True)

    final_triplets = final_triplets.drop_duplicates(subset=["subject", "predicate", "object", "date", "fragment_id", "speech_id"])
    if sort_time:
        final_triplets = final_triplets.sort_values(by=["date", "fragment_id"], ascending=[True, True]).reset_index(drop=False)

    final_triplets["index"] = final_triplets.index + 1

    main_triplets = final_triplets[final_triplets["information_type"] == "main"]
    additional_triplets = final_triplets[final_triplets["information_type"] == "additional"]

    prompt = prompt_template.format(
        question=question,
        main_facts=main_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records"),
        additional_facts=additional_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records")
    )

    return prompt, final_triplets

In [16]:
def get_main_plus_additional_fragments_prompt(question, main_triplets_df, additional_triplets_df, fragment_index_name, sort_time=True):
    prompt_template = """
You are a political scientist model. You are given a research question and a set of fragments from political speeches.
Each of fragments has a date associated with it, which indicates when the event described by the fragment occurred. The date is represented in the format YYYY-MM-DD.
The fragments contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.

Fragments are split into two categories: main and additional.
Main fragments (most important) are directly related to the research question. 
Additional fragments provide context or background information for entities present in the question.

Main Fragments:
{main_fragments}

Additional Fragments:
{additional_fragments}

Research question: {question}

Your task is to answer the research question based on the provided fragments.
    """
    final_triplets = pd.concat([main_triplets_df, additional_triplets_df], ignore_index=True)

    final_triplets = final_triplets.drop_duplicates(subset=["subject", "predicate", "object", "date", "fragment_id", "speech_id"])
    if sort_time:
        final_triplets = final_triplets.sort_values(by=["date", "fragment_id"], ascending=[True, True]).reset_index(drop=False)

    final_triplets["index"] = final_triplets.index + 1

    main_triplets = final_triplets[final_triplets["information_type"] == "main"]
    additional_triplets = final_triplets[final_triplets["information_type"] == "additional"]

    main_triplets["fragment_text"] = ""
    additional_triplets["fragment_text"] = ""
    for row in main_triplets.itertuples():
        ref_fragment = get_fragment(final_triplets, row.index, fragment_index_name)

        main_triplets.loc[row.Index, "fragment_text"] = ref_fragment

    for row in additional_triplets.itertuples():
        ref_fragment = get_fragment(final_triplets, row.index, fragment_index_name)

        additional_triplets.loc[row.Index, "fragment_text"] = ref_fragment

    main_triplets = main_triplets.drop_duplicates(subset=["fragment_text", "date"])
    additional_triplets = additional_triplets.drop_duplicates(subset=["fragment_text", "date"])
    prompt = prompt_template.format(
        question=question,
        main_fragments=main_triplets[["index", "fragment_text", "date"]].to_dict(orient="records"),
        additional_fragments=additional_triplets[["index", "fragment_text", "date"]].to_dict(orient="records")
    )

    return prompt, pd.concat([main_triplets, additional_triplets], ignore_index=True)

In [17]:
def get_subgraph_triplets_prompt(question, subgraph_triplets_df, sort_time=True):
    prompt_template = """
You are a political scientist model. You are given a research question and a set of facts from a knowledge graph.
Each of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.
Each of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.
The triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.

Facts:
{facts}

Research question: {question}

Your task is to answer the research question based on the provided facts. 
    """
    final_triplets = subgraph_triplets_df.drop_duplicates(subset=["subject", "predicate", "object", "date", "fragment_id", "speech_id"])
    if sort_time:
        final_triplets = final_triplets.sort_values(by=["date", "fragment_id"], ascending=[True, True]).reset_index(drop=False)

    final_triplets["index"] = final_triplets.index + 1

    prompt = prompt_template.format(
        question=question,
        facts=final_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records")
    )

    return prompt, final_triplets

In [18]:
def get_subgraph_fragments_prompt(question, subgraph_triplets_df, fragment_index_name, sort_time=True):
    prompt_template = """
You are a political scientist model. You are given a research question and a set of fragments from political speeches.
Each of fragments has a date associated with it, which indicates when the event described by the fragment occurred. The date is represented in the format YYYY-MM-DD.
The fragments contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.

Fragments: {fragments}

Research question: {question}

Your task is to answer the research question based on the provided fragments.
    """
    final_triplets = subgraph_triplets_df.drop_duplicates(subset=["subject", "predicate", "object", "date", "fragment_id", "speech_id"])
    if sort_time:
        final_triplets = final_triplets.sort_values(by=["date", "fragment_id"], ascending=[True, True]).reset_index(drop=False)

    final_triplets["index"] = final_triplets.index + 1

    for row in final_triplets.itertuples():
        ref_fragment = get_fragment(final_triplets, row.index, fragment_index_name)

        final_triplets.loc[row.Index, "fragment_text"] = ref_fragment
    final_triplets = final_triplets.drop_duplicates(subset=["fragment_text", "date"])
    try:
        fragments = final_triplets[["index", "fragment_text", "date"]].to_dict(orient="records")
    except Exception as e:
        fragments = []
    prompt = prompt_template.format(
        question=question,
        fragments=fragments
    )

    return prompt, final_triplets

In [19]:
def get_prompt(
        graph,
        question,
        triplets_index_name,
        ent_index_name,
        fragment_index_name,
        score_threshold=0.5,
        start_date="1900-01-01",
        end_date="2100-01-01"
    ):

    main_triplets_related_to_question = find_question_related_triplets(
        graph=graph,
        question=question,
        triplets_index_name=triplets_index_name,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_k=1000
    )
    additional_triplets_related_to_question = find_additional_related_triplets(
        graph=graph,
        question=question,
        triplets_index_name=triplets_index_name,
        ent_index_name=ent_index_name,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_entity_k=10,
        top_triplet_k=100
    )
    main_triplets_with_subgraph = find_subgraph_for_question(
        graph=graph,
        main_triplets_related_to_question=main_triplets_related_to_question,
        question=question,
        triplets_index_name=triplets_index_name,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_triplet_k=1000000
    )

    prompts = {}
    final_triplets_dict = {}
    # prompts["qpt+ts"], final_triplets_dict["main_plus_additional_triplets"] = get_main_plus_additional_triplets_prompt(
    #     question=question,
    #     main_triplets_df=main_triplets_related_to_question,
    #     additional_triplets_df=additional_triplets_related_to_question,
    #     sort_time=True
    # )

    # prompts["qpt+ss"], final_triplets_dict["main_plus_additional_triplets"] = get_main_plus_additional_triplets_prompt(
    #     question=question,
    #     main_triplets_df=main_triplets_related_to_question,
    #     additional_triplets_df=additional_triplets_related_to_question,
    #     sort_time=False
    # )

    # prompts["qpf+ts"], final_triplets_dict["main_plus_additional_fragments"] = get_main_plus_additional_fragments_prompt(
    #     question=question,
    #     main_triplets_df=main_triplets_related_to_question,
    #     additional_triplets_df=additional_triplets_related_to_question,
    #     fragment_index_name=fragment_index_name,
    #     sort_time=True
    # )

    # prompts["qpf+ss"], final_triplets_dict["main_plus_additional_fragments"] = get_main_plus_additional_fragments_prompt(
    #     question=question,
    #     main_triplets_df=main_triplets_related_to_question,
    #     additional_triplets_df=additional_triplets_related_to_question,
    #     fragment_index_name=fragment_index_name,
    #     sort_time=False
    # )

    # prompts["qpf+ts"], final_triplets_dict["main_plus_additional_fragments"] = get_main_plus_additional_fragments_prompt(
    #     question=question,
    #     main_triplets_df=main_triplets_related_to_question,
    #     additional_triplets_df=additional_triplets_related_to_question,
    #     fragment_index_name=fragment_index_name,
    #     sort_time=True
    # )

    # prompts["qpf+ss"], final_triplets_dict["main_plus_additional_fragments"] = get_main_plus_additional_fragments_prompt(
    #     question=question,
    #     main_triplets_df=main_triplets_related_to_question,
    #     additional_triplets_df=additional_triplets_related_to_question,
    #     fragment_index_name=fragment_index_name,
    #     sort_time=False
    # )

    # prompts["st+ts"], final_triplets_dict["subgraph_triplets"] = get_subgraph_triplets_prompt(
    #     question=question,
    #     subgraph_triplets_df=main_triplets_with_subgraph,
    #     sort_time=True
    # )

    # prompts["st+ss"], final_triplets_dict["subgraph_triplets"] = get_subgraph_triplets_prompt(
    #     question=question,
    #     subgraph_triplets_df=main_triplets_with_subgraph,
    #     sort_time=False
    # )

    prompts["sf+ts"], final_triplets_dict["subgraph_fragments"] = get_subgraph_fragments_prompt(
        question=question,
        subgraph_triplets_df=main_triplets_with_subgraph,
        fragment_index_name=fragment_index_name,
        sort_time=True
    )

    # prompts["sf+ss"], final_triplets_dict["subgraph_fragments"] = get_subgraph_fragments_prompt(
    #     question=question,
    #     subgraph_triplets_df=main_triplets_with_subgraph,
    #     fragment_index_name=fragment_index_name,
    #     sort_time=False
    # )

    return prompts, final_triplets_dict, main_triplets_related_to_question, additional_triplets_related_to_question, main_triplets_with_subgraph

In [20]:
def get_fragment(triplets_resp, idx, fragment_index_name):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    fragment_id = row.fragment_id
    return es_client.search(index=fragment_index_name, query={"match": {"_id": fragment_id}}, size=100)["hits"]["hits"][0]["_source"]["text"]

def get_speech(triplets_resp, idx, index_name):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    speech_id = row.speech_id

    return es_client.search(index=index_name, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"]

In [21]:
def prepare_prompts(question, start_date, end_date):

    # enhanced_prompts, enhanced_final_triplets_dict, enhanced_main_triplets_related_to_question, enhanced_additional_triplets_related_to_question, enhanced_main_triplets_with_subgraph = get_prompt(
    #     graph=graph_enhanced,
    #     question=question,
    #     triplets_index_name=TRIPLETS_INDEX_NAME_ENHANCED,
    #     ent_index_name=ENT_INDEX_NAME_ENHANCED,
    #     fragment_index_name=FRAGMENT_INDEX_NAME,
    #     score_threshold=0.5,
    #     start_date=start_date,
    #     end_date=end_date
    # )

    no_enhanced_prompts, not_enhanced_final_triplets_dict, not_enhanced_main_triplets_related_to_question, not_enhanced_additional_triplets_related_to_question, not_enhanced_main_triplets_with_subgraph = get_prompt(
        graph=graph_not_enhanced,
        question=question,
        triplets_index_name=TRIPLETS_INDEX_NAME_NOT_ENHANCED,
        ent_index_name=ENT_INDEX_NAME_NOT_ENHANCED,
        fragment_index_name=FRAGMENT_INDEX_NAME,
        score_threshold=0.5,
        start_date=start_date,
        end_date=end_date
    )

    final_prompts = {}

    # for ek, ev in enhanced_prompts.items():
    #     final_prompts[f"te+{ek}"] = ev

    for nk, nv in no_enhanced_prompts.items():
        final_prompts[f"nte+{nk}"] = nv

    return final_prompts

### Evaluation Questions for each issue

In [22]:
ANSWER_FORMATS = {
    "binary": "Answer Format: 'yes' or 'no'. Return only the answer. No other text.",
    "single_phrase": "Answer Format: return only a very short keyword or noun phrase, preferably 1-3 words. No other text.",
    "sentence": "Answer Format: Short sentence. No other text.",
    "date": "Answer Format: date in form of YYYY between 2000 and 2025. Return only the year. No other text.",
}

In [23]:
NLI_MODEL_NAME = "facebook/bart-large-mnli"

NLI_TOKENIZER = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
NLI_MODEL = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)

def nli_score(premise, hypothesis, l):
    inputs = NLI_TOKENIZER(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    with torch.no_grad():
        logits = NLI_MODEL(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    scores = {
        "contradiction": probs[0].item(),
        "neutral": probs[1].item(),
        "entailment": probs[2].item()
    }

    return scores["entailment"] + l * scores["neutral"]

In [24]:
def evaluate_answer(model_answer, gold_answer, question_type, alpha):
    if model_answer == "not_specified":
        if question_type == "binary":
            return 0.0
        if question_type == "single_phrase":
            return 0.0
        if question_type == "sentence":
            return 0.0
        if question_type == "date":
            return "-"
    if question_type == "binary":
        return 1.0 if model_answer.strip().replace(".", "").lower() == gold_answer.strip().lower() else 0.0
    elif question_type == "single_phrase":
        m_encoded = EMBEDDING_MODEL.encode(model_answer, normalize_embeddings=True)
        g_encoded = EMBEDDING_MODEL.encode(gold_answer, normalize_embeddings=True)
        return float(np.round(float(np.dot(m_encoded, g_encoded)), 4))

    elif question_type == "sentence":
        cap = alpha * nli_score(gold_answer, model_answer, l=0.3) + (1 - alpha) * nli_score(model_answer, gold_answer, l=0.3)
        return float(np.round(cap, 4))

    elif question_type == "date":
        try:
            model_answer_year = int(model_answer)
        except Exception as e:
            return "-"

        return abs(model_answer_year - int(gold_answer))

In [25]:
def run_single_experiment(df, prepared_prompts, print_results=False):
    results = {}

    for _, row in df.iterrows():
        final_prompts = prepared_prompts[row["question_idx"]]

        for k, v in final_prompts.items():
            if k not in results:
                results[k] = {}
            response = openai_client.responses.create(
                model="gpt-6-luna",
                reasoning={"effort": "low"},
                instructions=SOURCE_DISCLAIMER.format(answer_format=ANSWER_FORMATS[row['question_type']]),
                input=(f"{v}\n\n"),
                store=False,
            )

            res = response.output_text.strip()
            evaluation_score = evaluate_answer(
                model_answer=res,
                gold_answer=row["question_answer"],
                question_type=row["question_type"],
                alpha=0.75
            )

            if print_results:
                print(f"Model Answer: {res}, Evaluation Score: {evaluation_score}")

            results[k][row["question_idx"]] = evaluation_score

    return results

In [26]:
SOURCE_DISCLAIMER = """
Answer using ONLY the facts explicitly provided in the input.

Do not use prior knowledge, world knowledge, assumptions, or information
not contained in the supplied facts.

Treat the supplied facts as the complete source of evidence for the task.
If the supplied facts provide no reasonable basis for answering the question return exactly:

not_specified

Otherwise, follow the requested answer format exactly:

{answer_format}
""".strip()

def prepare_all_prompts(df):
    prepared_prompts = {}
    for _, row in df.iterrows():
        start_date, end_date = row["time_frame"].split(",")
        prepared_prompts[row["question_idx"]] = prepare_prompts(
            question=row["question"],
            start_date=start_date.strip(),
            end_date=end_date.strip()
        )
    return prepared_prompts

In [27]:
def run_question_experiment(df, n=3, print_results=False):

    prepared_prompts = prepare_all_prompts(df)
    sentence_indices = np.where(df["question_type"] == "sentence")[0]
    single_phrase_indices = np.where(df["question_type"] == "single_phrase")[0]
    binary_indices = np.where(df["question_type"] == "binary")[0]
    date_indices = np.where(df["question_type"] == "date")[0]

    type_indices = {
        "sentence": sentence_indices,
        "single_phrase": single_phrase_indices,
        "binary": binary_indices,
        "date": date_indices,
    }
    results = []

    for _ in tqdm(range(n), desc="Running experiments"):
        single_res = (
            pd.DataFrame.from_dict(
                run_single_experiment(
                    df,
                    prepared_prompts=prepared_prompts,
                    print_results=print_results
                ),
                orient="index"
            ).replace("-", np.nan).apply(pd.to_numeric, errors="coerce")
        )

        results.append(single_res)

    mean_df = (
        pd.concat(results, keys=range(len(results)))
        .groupby(level=1)
        .mean()
    )
    rows = list(mean_df.index)
    cols = list(mean_df.columns)
    arr = np.stack([df.loc[rows, cols].to_numpy(dtype=float)for df in results])

    std_sample = np.nanstd(arr, axis=0, ddof=1)
    std_df = pd.DataFrame(std_sample, index=rows, columns=cols)

    return results, std_df.round(3), mean_df.round(3)

1. How does the image of Ukraine change in his speeches from 2000 to 2024?

In [28]:
q1 = pd.DataFrame(
    {
        "question_idx": [
            "Q1","Q2","Q3","Q4","Q5","Q6","Q7","Q8","Q9","Q10","Q11","Q12","Q13"
        ],
        "question": [
            "How is Ukraine portrayed?",
            "How is Ukraine portrayed?",
            "How is Ukraine portrayed?",
            "What is the approach to Ukrainian civilians?",
            "What is the approach to Ukrainian civilians?",
            "What is the approach to Ukrainian state and elites?",
            "What is the approach to Ukrainian state and elites?",
            "Who is enemy of russia during a war?",
            "Is Ukraine portrayed as hostile whole time?",
            "Does the portrayal of Ukraine become more radical over time?",
            "What year marks the change from partnership to paternalism in the portrayal of Ukraine?",
            "What year marks the beginning of Ukraine being portrayed as taken over by the West?",
            "What year marks the change from paternalism to effective questioning of Ukraine's separate statehood?",

        ],
        "time_frame": [
            "2000-01-01,2004-01-01",
            "2008-01-01,2013-12-31",
            "2022-02-24,2030-12-31",
            "2014-01-01,2020-12-31",
            "2021-01-01,2030-12-31",
            "2014-01-01,2020-12-31",
            "2021-01-01,2030-12-31",
            "2014-01-01,2020-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
        ],
        "question_type": [
            "sentence",
            "sentence",
            "sentence",
            "single_phrase",
            "single_phrase",
            "single_phrase",
            "single_phrase",
            "single_phrase",
            "binary",
            "binary",
            "date",
            "date",
            "date",
        ],
        "question_answer": [
            "Ukraine is a neighbor, partner and its connected with Russia economically, historically, and culturally.",
            "Ukraine is having limited political agency and its not being fully separate from Russia.",
            "Ukraine is an anti-russia instrument of the west and a nato foothold",
            "brothers",
            "brothers",
            "delegitimized",
            "delegitimized",
            "west",
            "no",
            "yes",
            "2004",
            "2014",
            "2022"
        ]
    }
)

In [ ]:
q1_whole_results, q1_std_df, q1_mean_df = run_question_experiment(q1, n=10, print_results=False)

q1_mean_df.to_csv("exp_results/q1_results_mean.csv", index=True)
q1_std_df.to_csv("exp_results/q1_results_std.csv", index=True)
with open("exp_results/q1_whole_results.pkl", "wb") as f:
    pickle.dump(q1_whole_results, f)

2. When do themes of sovereignty begin to dominate foreign policy discourse?

In [29]:
q2 = pd.DataFrame(
    {
        "question_idx": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9", "Q10"],
        "question": [
            "In what context are sovereignty themes mentioned?",
            "In what context are sovereignty themes mentioned?",
            "In what context are sovereignty themes mentioned?",
            "What characteristic of foreign policy accompanies sovereignty?",
            "What forces the belief that the state must be resistant to external interference?",
            "Does USA influence sovereignty themes?",
            "Does sovereignty become less important after 2012?",
            "Does the amount of sovereignty themes increase over time?",
            "Which year marks the first increase in the importance of sovereignty themes?",
            "Which year marks the second increase in the importance of sovereignty themes?",
        ],
        "time_frame": [
            "2004-01-01,2007-12-31",
            "2012-01-01,2014-12-31",
            "2015-01-01,2030-12-31",
            "2000-01-01,2004-12-31",
            "2004-01-01,2007-12-31",
            "2000-01-01,2004-01-01",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
        ],
        "question_type": [
            "sentence",
            "sentence",
            "sentence",
            "single_phrase",
            "single_phrase",
            "binary",
            "binary",
            "binary",
            "date",
            "date"
        ],
        "question_answer": [
            "Sovereignty is important as the state must be resistant to external interference.",
            "Sovereignty is a political slogan.",
            "Sovereignty is one of the central concepts organizing entire approach to international politics.",
            "independence",
            "NATO reinforce",
            "yes",
            "no",
            "yes",
            "2004",
            "2012"
        ],
        
    }
)

In [30]:
q2_whole_results, q2_std_df, q2_mean_df= run_question_experiment(q2, n=10, print_results=False)

q2_mean_df.to_csv("exp_results/q2_results_mean.csv", index=True)
q2_std_df.to_csv("exp_results/q2_results_std.csv", index=True)
with open("exp_results/q2_whole_results.pkl", "wb") as f:
    pickle.dump(q2_whole_results, f)

Running experiments: 100%|██████████| 10/10 [39:56<00:00, 239.66s/it]


3. At what point does Putin start talking about a multipolar world?

In [30]:
q3 = pd.DataFrame(
    {
        "question_idx": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9", "Q10"],
        "question": [
            "How is multipolarity portrayed?",
            "How is multipolarity portrayed?",
            "How is international system portrayed in relation to multipolarity?",
            "Whose dominance is multipolarity initially meant to counterbalance?",
            "Which model does Putin support? Multipolar or unipolar?",
            "Does Putin describe the unipolar model as acceptable?",
            "Does the concept of multipolarity first appear near 2007?",
            "Does USA have influence on multipolar world term introduction?",
            "What year does multipolarity becomes one of the most recognizable elements of Putin's personal political message?",
            "What year does multipolarity start to appear?",
        ],
        "time_frame": [
            "2000-01-01,2007-01-01",
            "2010-01-01,2030-12-31",
            "2010-01-01,2030-12-31",
            "2000-01-01,2007-01-01",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2007-01-01",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
        ],
        "question_type": [
            "sentence",
            "sentence",
            "sentence",
            "single_phrase",
            "single_phrase",
            "binary",
            "binary",
            "binary",
            "date",
            "date"
        ],
        "question_answer": [
            "Multipolarity is a counterweight to American unipolarity.",
            "Multipolarity is a several major centers of power, Russia as one of them.",
            "International system is a process of negotiating interests among major powers.",
            "United States",
            "Multipolar",
            "no",
            "yes",
            "no",
            "2007",
            "2000"
        ]
    }
)

In [32]:
q3_whole_results, q3_std_df, q3_mean_df = run_question_experiment(q3, n=10, print_results=False)

q3_mean_df.to_csv("exp_results/q3_results_mean.csv", index=True)
q3_std_df.to_csv("exp_results/q3_results_std.csv", index=True)
with open("exp_results/q3_whole_results.pkl", "wb") as f:
    pickle.dump(q3_whole_results, f)

Running experiments: 100%|██████████| 10/10 [33:33<00:00, 201.35s/it]


4. Is Russia more often described as a victim, a leader, or a defender?

In [31]:
q4 = pd.DataFrame(
    {
        "question_idx": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9", "Q10", "Q11", "Q12", "Q13", "Q14"],
        "question": [
            "How is Russia portrayed as a victim?",
            "How is Russia portrayed as a defender?",
            "How is Russia portrayed as a leader or major power?",
            "What are justifications of Russia's offensive war?",
            "How are the victim, defender, and leader images related?",
            "What kind of values does Russia claim to defend?",
            "What military status does Russia have?",
            "What is Russia status on international stage?",
            "Is Russia portrayed as a victim of West?",
            "Is Russia portrayed as a military leader?",
            "Is Russia also portrayed as a powerful international actor?",
            "Is Russia portrayed as a defender of its own interests?",
            "Is Russia portrayed as waging war without justification?",
            "Are the victim and leader images of Russia mutually exclusive?"
        ],
        "time_frame": [
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2022-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
        ],
        "question_type": [
            "sentence",
            "sentence",
            "sentence",
            "sentence",
            "sentence",
            "single_phrase",
            "single_phrase",
            "single_phrase",
            "binary",
            "binary",
            "binary",
            "binary",
            "binary",
            "binary",
        ],
        "question_answer": [
            "Russia is portrayed as a victim of NATO enlargement, the disregard of Russian interests, Western double standards, color revolutions, and sanctions.",
            "Russia is portrayed as a defender of itself, Russians, the Russian world, and traditional values.",
            "Russia is portrayed as the victor of the Second World War, a nuclear power, and one of the major poles of the world.",
            "Russia is portrayed as waging a defensive war against NATO, neo-Nazism, and alleged attempts to destroy Russia.",
            "Victim, Defender, and Leader images are interconnected.",
            "traditional values",
            "nuclear power",
            "pole of the world",
            "yes",
            "yes",
            "yes",
            "yes",
            "no",
            "no",
        ]
    }
)

In [ ]:
q4_whole_results, q4_std_df, q4_mean_df = run_question_experiment(q4, n=10, print_results=False)

q4_mean_df.to_csv("exp_results/q4_results_mean.csv", index=True)
q4_std_df.to_csv("exp_results/q4_results_std.csv", index=True)
with open("exp_results/q4_whole_results.pkl", "wb") as f:
    pickle.dump(q4_whole_results, f)

5. How does the narrative toward Poland differ from that toward Germany?

In [32]:
q5 = pd.DataFrame(
    {
        "question_idx": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9", "Q10", "Q11", "Q12", "Q13"],
        "question": [
            "How is Poland portrayed?",
            "How is Germany portrayed?",
            "What historical events are used in Poland context?",
            "What role is Poland assigned in relation to Russia?",
            "What historical role is Germany associated with?",
            "Is Germany portrayed only as a historical enemy?",
            "Is Germany portrayed as an important economic and political partner?",
            "Is Poland portrayed as a privileged partner of Russia?",
            "Is Poland associated with Russophobia?",
            "Are Poland and Germany portrayed differently?",
            "Does portrayal of Germany change over time?",
            "At what year the major escalation in historical criticism of Poland started?",
            "At what year the difference between the portrayals of Poland and Germany becomes smaller?"
        ],
        "time_frame": [
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
        ],
        "question_type": [
            "sentence",
            "sentence",
            "sentence",
            "single_phrase",
            "single_phrase",
            "binary",
            "binary",
            "binary",
            "binary",
            "binary",
            "binary",
            "date",
            "date",
        ],
        "question_answer": [
            "Poland is portrayed as a Russophobic country, a political opponent of Russia, and a supporter of NATO enlargement.",
            "Germany is portrayed as a historical aggressor, but also as an economic and political partner.",
            "Poland is portrayed as sharing responsibility for the crisis that led to the Second World War, as marked by antisemitism in the Second Polish Republic, while the Soviet interpretation of the Molotov-Ribbentrop Pact is defended.",
            "political opponent",
            "nazi aggressor",
            "no",
            "yes",
            "no",
            "yes",
            "yes",
            "yes",
            "2019",
            "2022",
        ]
    }
)

In [ ]:
q5_whole_results, q5_std_df, q5_mean_df = run_question_experiment(q5, n=10, print_results=False)

q5_mean_df.to_csv("exp_results/q5_results_mean.csv", index=True)
q5_std_df.to_csv("exp_results/q5_results_std.csv", index=True)
with open("exp_results/q5_whole_results.pkl", "wb") as f:
    pickle.dump(q5_whole_results, f)

Running experiments: 100%|██████████| 10/10 [45:35<00:00, 273.51s/it]


6. How does the tone toward the USA compare with that toward the European Union?

In [33]:
q6 = pd.DataFrame(
    {
        "question_idx": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9", "Q10"],
        "question": [
            "How is the United States portrayed?",
            "How is the European Union portrayed?",
            "Why European Union began to be portrayed as a threat?",
            "What country is portrayed as a leader of the collective West?",
            "Which country is portrayed as the main force behind NATO enlargement?",
            "What role is Europe assigned economically?",
            "Is the United States overally portrayed more negatively than the European Union?",
            "Does portrayal of the European Union become more negative over time?",
            "When did the European Union begin to be portrayed as a threat?",
            "When did the United States, NATO, and the EU start to being increasingly merged into the broader category of the collective West?"
            
        ],
        "time_frame": [
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
            "2000-01-01,2030-12-31",
        ],
        "question_type": [
            "sentence",
            "sentence",
            "single_phrase",
            "single_phrase",
            "single_phrase",
            "single_phrase",
            "binary",
            "binary",
            "date",
            "date"
        ],
        "question_answer": [
            "The United States is portrayed as the main force behind NATO enlargement, color revolutions, and unipolarity.",
            "While the European Union is portrayed as a part of the West, it is also seen as an economic partner and a culturally close region.",
            "sanctions",
            "United States",
            "United States",
            "economic partner",
            "yes",
            "yes",
            "2013",
            "2022",
        ]
    }
)

In [ ]:
q6_whole_results, q6_std_df, q6_mean_df = run_question_experiment(q6, n=10, print_results=False)

q6_mean_df.to_csv("exp_results/q6_results_mean.csv", index=True)
q6_std_df.to_csv("exp_results/q6_results_std.csv", index=True)
with open("exp_results/q6_whole_results.pkl", "wb") as f:
    pickle.dump(q6_whole_results, f)

Running experiments: 100%|██████████| 10/10 [49:49<00:00, 298.90s/it]


### Aggregate scores

In [34]:
def combine_mean_std(means_df, std_df):
    combined_df = means_df.copy()
    for col in means_df.columns:
        combined_df[col] = [
            (
                f"{mean}±{std}"
                if pd.notna(mean) and pd.notna(std)
                else np.nan
            )
            for mean, std in zip(means_df[col], std_df[col])
        ]

    return combined_df

In [35]:
with open("exp_results/q1_whole_results.pkl", "rb") as f:
    q1_whole_results = pickle.load(f)
with open("exp_results/q2_whole_results.pkl", "rb") as f:
    q2_whole_results = pickle.load(f)
with open("exp_results/q3_whole_results.pkl", "rb") as f:
    q3_whole_results = pickle.load(f)
with open("exp_results/q4_whole_results.pkl", "rb") as f:
    q4_whole_results = pickle.load(f)
with open("exp_results/q5_whole_results.pkl", "rb") as f:
    q5_whole_results = pickle.load(f)
with open("exp_results/q6_whole_results.pkl", "rb") as f:
    q6_whole_results = pickle.load(f)

q1_mean_df = pd.read_csv("exp_results/q1_results_mean.csv", index_col=0)
q1_std_df = pd.read_csv("exp_results/q1_results_std.csv", index_col=0)

q2_mean_df = pd.read_csv("exp_results/q2_results_mean.csv", index_col=0)
q2_std_df = pd.read_csv("exp_results/q2_results_std.csv", index_col=0)

q3_mean_df = pd.read_csv("exp_results/q3_results_mean.csv", index_col=0)
q3_std_df = pd.read_csv("exp_results/q3_results_std.csv", index_col=0)

q4_mean_df = pd.read_csv("exp_results/q4_results_mean.csv", index_col=0)
q4_std_df = pd.read_csv("exp_results/q4_results_std.csv", index_col=0)

q5_mean_df = pd.read_csv("exp_results/q5_results_mean.csv", index_col=0)
q5_std_df = pd.read_csv("exp_results/q5_results_std.csv", index_col=0)

q6_mean_df = pd.read_csv("exp_results/q6_results_mean.csv", index_col=0)
q6_std_df = pd.read_csv("exp_results/q6_results_std.csv", index_col=0)

question_results = [
    (q1, q1_whole_results, q1_mean_df, q1_std_df),
    (q2, q2_whole_results, q2_mean_df, q2_std_df),
    (q3, q3_whole_results, q3_mean_df, q3_std_df),
    (q4, q4_whole_results, q4_mean_df, q4_std_df),
    (q5, q5_whole_results, q5_mean_df, q5_std_df),
    (q6, q6_whole_results, q6_mean_df, q6_std_df)
]

In [36]:
sentence_mean_df = pd.DataFrame()
single_phrase_mean_df = pd.DataFrame()
binary_mean_df = pd.DataFrame()
date_mean_df = pd.DataFrame()

sentence_std_df = pd.DataFrame()
single_phrase_std_df = pd.DataFrame()
binary_std_df = pd.DataFrame()
date_std_df = pd.DataFrame()

In [37]:
for i in range(len(question_results)):
    question_df, _, mean_df, std_df = question_results[i]
    question_types = question_df["question_type"].values
    for j, q_type in enumerate(question_types):
        if q_type == "sentence":
            sentence_mean_df[f"T{i+1}_Q{j+1}"] = mean_df.iloc[:, j]
            sentence_std_df[f"T{i+1}_Q{j+1}"] = std_df.iloc[:, j]
        elif q_type == "single_phrase":
            single_phrase_mean_df[f"T{i+1}_Q{j+1}"] = mean_df.iloc[:, j]
            single_phrase_std_df[f"T{i+1}_Q{j+1}"] = std_df.iloc[:, j]
        elif q_type == "binary":
            binary_mean_df[f"T{i+1}_Q{j+1}"] = mean_df.iloc[:, j]
            binary_std_df[f"T{i+1}_Q{j+1}"] = std_df.iloc[:, j]
        elif q_type == "date":
            date_mean_df[f"T{i+1}_Q{j+1}"] = mean_df.iloc[:, j]
            date_std_df[f"T{i+1}_Q{j+1}"] = std_df.iloc[:, j]

upgraded_index = ['nte+qp+f+ss', 'nte+qp+f+ts', 'nte+qp+t+ss', 'nte+qp+t+ts', 'nte+s+f+ss', 'nte+s+f+ts', 'nte+s+t+ss', 'nte+s+t+ts', 'te+qp+f+ss', 'te+qp+f+ts', 'te+qp+t+ss', 'te+qp+t+ts', 'te+s+f+ss', 'te+s+f+ts', 'te+s+t+ss', 'te+s+t+ts']
sentence_mean_df.index = upgraded_index
single_phrase_mean_df.index = upgraded_index
binary_mean_df.index = upgraded_index
date_mean_df.index = upgraded_index
sentence_std_df.index = upgraded_index
single_phrase_std_df.index = upgraded_index
binary_std_df.index = upgraded_index
date_std_df.index = upgraded_index

In [38]:
# combined = combine_mean_std(date_mean_df, date_std_df)
# display(Markdown(combined.to_markdown()))

In [39]:
final_sentence_mean_df = deepcopy(sentence_mean_df).iloc[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 15, 17, 18]]
final_single_phrase_mean_df = deepcopy(single_phrase_mean_df).iloc[:, [1, 2, 5, 8, 9, 11, 13, 15, 17]]
final_binary_mean_df = deepcopy(binary_mean_df).iloc[:, [0, 1, 2, 8, 9, 10, 11, 12, 14, 15, 19, 20]]
final_date_mean_df = deepcopy(date_mean_df).iloc[:, [0, 2, 3, 4, 6, 9]]

In [40]:
df_index = list(final_sentence_mean_df.index)
exp_doubles = []
for i in range(len(df_index)):
    for j in range(i, len(df_index)):
        a = df_index[i].split("+")
        b = df_index[j].split("+")
        same_count = 0
        diff_index = -1
        for k in range(len(a)):
            if a[k] == b[k]:
                same_count += 1
            else:
                diff_index = k
        if same_count == 3:
            exp_doubles.append((df_index[i], df_index[j], diff_index))

exp_mapping = {
    "nte > te": [],
    "qp > s": [],
    "f > t": [],
    "ss > ts": []
}
exp_mapping_keys = list(exp_mapping.keys())
for i in exp_doubles:
    exp_mapping[exp_mapping_keys[i[2]]].append((i[0], i[1]))

df_exp_mapping = {
    "sentence": sentence_mean_df,
    "single_phrase": single_phrase_mean_df,
    "binary": binary_mean_df,
    "date": date_mean_df
}

In [41]:
results_from_all_exps = {}

for exp_name, exp_pairs in exp_mapping.items():
    single_exp_results = {}

    for test_name, test_df in df_exp_mapping.items():
        avgs_for_df = []
        for conf1, conf2 in exp_pairs:
            a = test_df.loc[conf1]
            b = test_df.loc[conf2]
            if test_name == "date":
                mask = a.notna() & b.notna()
                a_fixed = a.fillna(np.inf)
                b_fixed = b.fillna(np.inf)
                avgs_for_df.extend(list(1*(a_fixed[mask] > b_fixed[mask])))

            else:
                mask = a != b
                avgs_for_df.extend(list(1*(a[mask] > b[mask])))

        single_exp_results[test_name] = round(float(np.mean(avgs_for_df)), 2)

    results_from_all_exps[exp_name] = single_exp_results


In [42]:
results_from_all_exps

{'nte > te': {'sentence': 0.61,
  'single_phrase': 0.56,
  'binary': 0.55,
  'date': 0.21},
 'qp > s': {'sentence': 0.46,
  'single_phrase': 0.39,
  'binary': 0.3,
  'date': 0.15},
 'f > t': {'sentence': 0.73,
  'single_phrase': 0.67,
  'binary': 0.78,
  'date': 0.36},
 'ss > ts': {'sentence': 0.45,
  'single_phrase': 0.47,
  'binary': 0.29,
  'date': 0.29}}

### Prepare a single gold dataset

In [ ]:
q1["task_idx"] = "Task1"
q2["task_idx"] = "Task2"
q3["task_idx"] = "Task3"
q4["task_idx"] = "Task4"
q5["task_idx"] = "Task5"
q6["task_idx"] = "Task6"

question_df_merged = pd.concat([q1, q2, q3, q4, q5, q6], ignore_index=True)
col_data = question_df_merged.pop("task_idx")
question_df_merged.insert(0, "task_idx", col_data)

question_df_merged.to_csv("question_df_merged.csv", index=False)

### Tests on larger models

In [74]:
# Trimm full gold df to delete questions with high variance as defined previously

full_gold_df = pd.read_csv("ref_exp_results_on_whole_gold_dataset/gold_dataset.csv")
with open("notebooklm_answers.json", "r") as f:
    notebooklm_answers = json.load(f)
all_dfs = [
    (sentence_mean_df, final_sentence_mean_df),
    (single_phrase_mean_df, final_single_phrase_mean_df),
    (binary_mean_df, final_binary_mean_df),
    (date_mean_df, final_date_mean_df)
]
for i in all_dfs:
    for j in list(i[0].columns):
        if j not in i[1].columns:
            task_idx = f"Task{j.split('_')[0][1]}"
            quesiton_idx = f"{j.split('_')[1]}"
            full_gold_df = full_gold_df[~((full_gold_df["task_idx"] == task_idx) & (full_gold_df["question_idx"] == quesiton_idx))]
            notebooklm_answers[task_idx].pop(quesiton_idx, None)
full_gold_df.to_csv("gold_dataset_trimmed.csv", index=False)
with open("notebooklm_answers.json", "w") as f:
    json.dump(notebooklm_answers, f, indent=4)

In [75]:
gold_df = pd.read_csv("gold_dataset_trimmed.csv")

In [100]:
REFERENCE_PROMPT_TEMPLATE = """Based ONLY on sources I provided answer the question: {question} between {start_date} and {end_date}?"""

prompt_df = pd.DataFrame(columns=["task_idx", "question_idx", "question_type", "my_prompt", "reference_prompt"])

for idx, row in gold_df.iterrows():
    start_date, end_date = row["time_frame"].split(",")
    prompt = prepare_prompts(
        question=row["question"],
        start_date=start_date,
        end_date=end_date
    )["nte+sf+ts"]
    disclaimer = SOURCE_DISCLAIMER.format(answer_format=ANSWER_FORMATS[row['question_type']])
    my_prompt = prompt + "\n\n" + disclaimer
    reference_prompt = REFERENCE_PROMPT_TEMPLATE.format(
        question=row["question"].replace("?", ""),
        start_date=start_date,
        end_date=end_date
    ) + " " + ANSWER_FORMATS[row["question_type"]]
    prompt_df.loc[idx] = [row["task_idx"], row["question_idx"], row["question_type"], my_prompt, reference_prompt]

In [106]:
sol_answers = json.load(open("sol_answers.json", "r"))
notebooklm_answers = json.load(open("notebooklm_answers.json", "r"))

In [103]:
for idx, row in tqdm(prompt_df.iterrows(), "Generating answers..."):
    task_idx = row["task_idx"]
    question_idx = row["question_idx"]
    question_type = row["question_type"]
    if not sol_answers.get(task_idx):
        sol_answers[task_idx] = {}

    response = openai_client.responses.create(
        model="gpt-5.6-sol",
        reasoning={"effort": "medium"},
        instructions=SOURCE_DISCLAIMER.format(answer_format=ANSWER_FORMATS[question_type]),
        input=row["my_prompt"],
        store=False,
    ).output_text.strip()

    sol_answers[task_idx][question_idx] = response

Generating answers...: 42it [01:40,  2.39s/it]


In [94]:
idx = 0

task_idx = prompt_df.loc[idx, "task_idx"]
question_idx = prompt_df.loc[idx, "question_idx"]
question_type = prompt_df.loc[idx, "question_type"]

if not sol_answers.get(task_idx):
    sol_answers[task_idx] = {}
# if not notebooklm_answers.get(task_idx):
#     notebooklm_answers[task_idx] = {}

In [86]:
print(prompt_df.iloc[idx, :]["my_prompt"])


You are a political scientist model. You are given a research question and a set of fragments from political speeches.
Each of fragments has a date associated with it, which indicates when the event described by the fragment occurred. The date is represented in the format YYYY-MM-DD.
The fragments contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.

Fragments: [{'index': 1, 'fragment_text': 'We may face instances of violence and armed clashes for a long time yet. But it will be a far cry from what we faced when large units numbering several thousand people invaded Dagestan. You know, what we have managed to do together is we have managed to bring home to every citizen the tragic character of the events that are happening and their importance for the fate of the country. And you would agree that only recently, last summer, it was hard for us to predict how the public would reac

In [ ]:
print(prompt_df.iloc[idx, :]["reference_prompt"])

In [84]:
sol_answers[task_idx][question_idx] = """Ukraine is portrayed as a sovereign, independent, and culturally close partner of Russia, with strong historical ties, shared interests, and extensive economic, political, and security cooperation."""
with open("sol_answers.json", "w", encoding="utf-8") as file:
    json.dump(sol_answers, file, ensure_ascii=False, indent=4)

In [ ]:
# notebooklm_answers[task_idx][question_idx] = """2022"""
# with open("notebooklm_answers.json", "w", encoding="utf-8") as file:
#     json.dump(notebooklm_answers, file, ensure_ascii=False, indent=4)

In [107]:
sol_results = []
notebooklm_results = []

for idx, row in gold_df.iterrows():
    try:
        sol_answer = sol_answers[row["task_idx"]][row["question_idx"]]
        notebooklm_answer = notebooklm_answers[row["task_idx"]][row["question_idx"]]
    except KeyError:
        print(f"Missing answer for task {row['task_idx']} and question {row['question_idx']}. Ending evaluation.")
        break

    sol_results.append(evaluate_answer(
        model_answer=sol_answer,
        gold_answer=row["question_answer"],
        question_type=row["question_type"],
        alpha=0.75
    ))

    notebooklm_results.append(evaluate_answer(
        model_answer=notebooklm_answer,
        gold_answer=row["question_answer"],
        question_type=row["question_type"],
        alpha=0.75
    ))

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


In [108]:
referential_scores = gold_df.iloc[:len(sol_results),:][["task_idx", "question_idx", "question_type"]].copy()
referential_scores["sol_scores"] = sol_results
referential_scores["notebooklm_scores"] = notebooklm_results

In [110]:
referential_scores.groupby("question_type")[["sol_scores", "notebooklm_scores"]].mean().round(3).T

question_type,binary,date,sentence,single_phrase
sol_scores,1.0,7.167,0.325,0.602
notebooklm_scores,1.0,4.167,0.297,0.560
